# Carbon pricing and Planetery boundaries

## Model

This notebook provides a numerical simulation of the effect of a potential carbon tax on variables which have been identified as 
key drivers of the processes underlying the planetary boundaries (PB) framework. The model, upon which we base our analysis, is set up along the lines of a standard economic textbook model of decentralized competitive economy where firms and consumers have preferences over outcomes and each solve maximization problems by engaging in trade with each other. The sectors (or economic actors) we have included are those that we have identified as being central drivers of the processes underlying the PB's these include e.g. the production of fossil fuel, fertilizer, agriculture, timber, water and fisheries.  To study the implications of a carbon tax within this setting we have proceeded along the following lines:
1. Characterize the problem each economic actor is trying to solve given their preferences, technology and constraints.
2. Derive the corresponding first order conditions for each actor.
3. From the first order conditions characterize the equilibrium allocation only in terms of the model variables (i.e. eliminate prices) 
4. Derive the comparitive statics of the effect of a carbon tax i.e. differentiate the equilibrium allocation fully with respect to the carbon tax.

The resulting equilibrium outcome thus indirectly reveals how each variable in the model responds to a carbon tax.

## Numerical solution method

The numerical solution is found as the solution to a system of 19 equations in 19 unknowns. This details of this equation system and how it was derived is found in the document referenced above (see equations 72-90). The solution method adopted below proceeds as follows:
1. From the equation system (72-90) all terms containing the carbon tax $\tau_E$ are moved to the left and all other terms to the right.
2. We then proceeded by rewriting the system in matrix/vector form where the rows correspond to the equations and columns are variables (e.g. the first row in the matrix holds the parameters of equation (72), the second row equation (73), etc.). The system can thus be written as $\mathbf{\tau} = \mathbf{A y}$ were $\mathbf{\tau}$ is a vector of containing the terms with the carbon tax, $\mathbf{y}$ are 19x1 vectors of outcome variables and $\mathbf{A}$ is a 19x19 matrix of parameters.
3. The solution to the system is thus found by inverting the matrix $\mathbf{A}$ and multiplying each side by this inverse i.e. $\mathbf{y} = \mathbf{A^{-1} \tau}$

## Code

All results can be produced in the paper can be generated by running by highlighting each code cell in this notebook and clicking the "run" for each cell  
in consecutive order. Further comments are provided in conjunction to the cells below. 

The web_model directory contains the code for solving the model (model.py), generating latex tables (pb_table_generator.py) and parameters (params.py).

## Requirements

Note: This code has been tested on a Mac version 10.14.4 with Python 3.7, Pandas version 0.23.4 and Numpy version 1.15.1. Auxillary Pandas and Numpy packages are not included in the standard library and must be installed seperately. 

In [1]:
# import necessary packages
import numpy as np
import pandas as pd
import re, importlib
from tqdm import tqdm
import os, sys
from itertools import product
from web_model import params, model  # Load parameters and model implementation.

importlib.reload(model)
pd.options.display.float_format = "{:20,.4f}".format
print(pd.__version__)
print(np.__version__)
# Note: Code has been tested with with Pandas version 0.23.4 and Numpy version 1.15.1

1.5.2
1.23.5


### Load params and model

In [2]:
param_list = list(params.df_typing_formatting.T.to_dict().values())
model_params = {row["keys"]: row["values"] for row in param_list}
sm = model.SolveModel(model_params)

### Solve model for CO2 tax only

In [3]:
df_carbontax_quantities, df_carbontax_prices = sm.gen_results(
    robust_check=False, biofuel_tax=0
)

pb = sm.pb_effects(df_carbontax_quantities)
# pb

In [4]:
pb.columns

Index(['description', 'max value', 'min value', 'mean value', 'Aerosol effect',
       'CO2 effect', 'Biodiv. incl. climate effect', 'Biogeochem. effect',
       'Freshwater effect', 'Ocean acid. effect', 'Land-use effect',
       'Ozone effect', 'Chem. effect'],
      dtype='object')

In [24]:
prod_change_dict = sm.prod_change_dict.copy()
prod_change_dict

{'A_LA': 0,
 'A_EpsA': 0,
 'A_P': 0,
 'A_W': 1,
 'A_MA': 1,
 'P_EP': 0,
 'P_Pho': 0,
 'P_MP': 0,
 'Eps_AB': 1,
 'Eps_EEps': 0,
 'Eps_R': 1,
 'Y_EpsY': 0,
 'Y_MY': 1,
 'Fi_EFi': 0,
 'Fi_MFi': 1,
 'T_LT': 1,
 'T_MT': 0}

In [29]:
def generate_combinations(num_variables, discrete_vals=[0, 1]):
    # Generate all possible combinations of 0 and 1 for the given number of variables
    combinations = list(product(discrete_vals, repeat=num_variables))
    return combinations

In [7]:
# Test all combinations of 1% change how efffects PB´s


def generate_combinations(num_variables, discrete_vals=[0, 1]):
    # Generate all possible combinations of 0 and 1 for the given number of variables
    combinations = list(product(discrete_vals, repeat=num_variables))
    return combinations


prod_change_dict_keys = list(sm.prod_change_dict.keys())

good_for_pb_combos = []
good_for_pb_combos_exclude_ozone_chem = []

num_variables = len(sm.prod_change_dict)
combinations = generate_combinations(num_variables)
print("Number of combinations: ", len(combinations))
for combo in tqdm(combinations[1:], total=len(combinations)):

    sm.prod_change_dict = dict(zip(prod_change_dict_keys, combo))
    df_carbontax_quantities, df_carbontax_prices = sm.gen_results(
        robust_check=False, biofuel_tax=0
    )
    pb = sm.pb_effects(df_carbontax_quantities)
    pb_sum = pb[pb.columns[4:]].sum()
    if all([s < 0 for s in list(pb_sum)]):
        good_for_pb_combos.append(combo)
        print(combo)
    elif all([s < 0 for s in list(pb_sum)[0:7]]):
        good_for_pb_combos_exclude_ozone_chem.append(combo)
    print(combo)
    print(pb_sum)
    break

Number of combinations:  131072


  0%|          | 0/131072 [00:00<?, ?it/s]

(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1)
Aerosol effect                               0.0001
CO2 effect                                   0.0036
Biodiv. incl. climate effect                 0.1574
Biogeochem. effect                          -0.0014
Freshwater effect                           -0.0004
Ocean acid. effect                           0.0036
Land-use effect                             -0.0197
Ozone effect                                 2.0000
Chem. effect                                -1.0000
dtype: float64


In [6]:
prod_change_dict_keys = list(sm.prod_change_dict.keys())

for pc in good_for_pb_combos_exclude_ozone_chem:

    sm.prod_change_dict = dict(zip(prod_change_dict_keys, pc))
    df_carbontax_quantities, df_carbontax_prices = sm.gen_results(
        robust_check=False, biofuel_tax=0
    )
    pb = sm.pb_effects(df_carbontax_quantities)
    pb_sum = pb[pb.columns[4:]].sum()
    pb_sum["Ozone effect"] = int(pb_sum["Ozone effect"] > 0) + -1 * int(
        pb_sum["Ozone effect"] < 0
    )
    pb_sum["Chem. effect"] = int(pb_sum["Chem. effect"] > 0) + -1 * int(
        pb_sum["Chem. effect"] < 0
    )

    print(pb_sum)
    print(sm.prod_change_dict)
    break

Aerosol effect                              -0.0100
CO2 effect                                   0.0390
Biodiv. incl. climate effect                 0.6384
Biogeochem. effect                          -0.6520
Freshwater effect                            0.0298
Ocean acid. effect                           0.0390
Land-use effect                             -0.3008
Ozone effect                                 1.0000
Chem. effect                                -1.0000
dtype: float64
{'A_LA': 1, 'A_EpsA': 1, 'A_P': 1, 'A_W': 1, 'A_MA': 1, 'P_EP': 1, 'P_Pho': 1, 'P_MP': 1, 'Eps_AB': 1, 'Eps_EEps': 1, 'Eps_R': 1, 'Y_EpsY': 1, 'Y_MY': 1, 'Fi_EFi': 1, 'Fi_MFi': 1, 'T_LT': 1, 'T_MT': 1}


In [12]:
import dataframe_image as dfi
import matplotlib
from matplotlib.colors import LinearSegmentedColormap

# Calculate how each resp tech change efffects PB´s
prod_change = {}
for k, v in sm.prod_change_dict.items():
    sm.prod_change_dict[k] = 1
    df_carbontax_quantities, df_carbontax_prices = sm.gen_results(
        robust_check=False, biofuel_tax=0
    )
    pb = sm.pb_effects(df_carbontax_quantities)
    sm.prod_change_dict[k] = 0
    pb_sum = pb[pb.columns[4:]].sum()
    pb_sum["Ozone effect"] = int(pb_sum["Ozone effect"] > 0) + -1 * int(
        pb_sum["Ozone effect"] < 0
    )
    pb_sum["Chem. effect"] = int(pb_sum["Chem. effect"] > 0) + -1 * int(
        pb_sum["Chem. effect"] < 0
    )
    tex_key = sm.prod_change_tex[k]
    prod_change[k] = pb_sum
df = pd.DataFrame(prod_change).T


def background_with_norm(s):

    cmap = matplotlib.cm.get_cmap("RdYlGn")
    c = [
        "darkred",
        "red",
        "lightcoral",
        "white",
        "palegreen",
        "green",
        "darkgreen",
    ][::-1]
    v = [0, 0.15, 0.4, 0.5, 0.6, 0.9, 1.0]
    l = list(zip(v, c))
    cmap = LinearSegmentedColormap.from_list("rg", l, N=256)

    def ind(c):
        if c >= 1:
            return cmap(np.array(1))
        elif c <= -1:
            return cmap(np.array(0))
        else:
            return cmap(np.array(0.5))

    vals = df.values.flatten()
    norm = matplotlib.colors.TwoSlopeNorm(vmin=vals.min(), vcenter=0, vmax=vals.max())
    if s.name in ["Ozone effect", "Chem. effect"]:

        def ctext(c):
            if c == 0.5:
                return "#A9A9A9"  # matplotlib.colors.to_hex(np.array([0.1, 0.1, 0.3]))
            return matplotlib.colors.to_hex(cmap(c.flatten()))

        return [
            "background-color: white; color: {:s}".format(ctext(c))
            for c in norm(s.values)
        ]

    else:
        norm = matplotlib.colors.TwoSlopeNorm(
            vmin=vals.min(), vcenter=0, vmax=vals.max()
        )
        return [
            "background-color: {:s}; color: #A9A9A9".format(
                matplotlib.colors.to_hex(c.flatten())
            )
            for c in cmap(norm(s.values))
        ]


df["Ozone effect"] = df[["Ozone effect"]].astype(int)
df["Chem. effect"] = df[["Chem. effect"]].astype(int)
dfs = df.style.apply(background_with_norm).format(
    "{:,}", subset=["Ozone effect", "Chem. effect"]
)

dfi.export(dfs, "styled_table.png")
dfs

167442 bytes written to file /var/folders/wm/1wm41qp50z96rm0pd9dz7t600000gn/T/tmpj5z8ck3x/temp.png


,Aerosol effect,CO2 effect,Biodiv. incl. climate effect,Biogeochem. effect,Freshwater effect,Ocean acid. effect,Land-use effect,Ozone effect,Chem. effect
A_LA,0.001067,0.041064,0.036063,-0.090569,-0.042128,0.041064,-0.009458,1,0
A_EpsA,0.000281,0.008976,0.011771,0.035264,0.018521,0.008976,0.021025,1,0
A_P,0.000203,0.006590,-0.006777,-0.740213,0.022800,0.006590,0.026027,1,0
A_W,0.000099,0.003172,0.004160,0.012461,-0.384285,0.003172,0.007430,1,0
A_MA,0.003885,0.124233,0.162914,0.488044,0.256321,0.124233,0.290978,1,0
P_EP,-0.000240,-0.007526,0.003817,-0.743812,0.001534,-0.007526,0.002009,1,-1
P_Pho,0.000143,0.004562,0.015670,-0.476585,0.006873,0.004562,0.007762,1,0
P_MP,0.000300,0.009553,0.032815,0.480183,0.014393,0.009553,0.016256,1,0
Eps_AB,-0.000611,-0.012469,-0.002055,0.025602,0.009821,-0.012469,-0.034703,-1,-1
Eps_EEps,-0.010602,-0.182229,0.006492,0.030007,-0.018279,-0.182229,0.023996,0,-1


In [22]:
df * np.array([2] * 17).reshape(17, 1)  # .sum(axis=0)

,Aerosol effect,CO2 effect,Biodiv. incl. climate effect,Biogeochem. effect,Freshwater effect,Ocean acid. effect,Land-use effect,Ozone effect,Chem. effect
A_LA,0.0021,0.0821,0.0721,-0.1811,-0.0843,0.0821,-0.0189,2,0
A_EpsA,0.0006,0.0180,0.0235,0.0705,0.0370,0.0180,0.0420,2,0
A_P,0.0004,0.0132,-0.0136,-1.4804,0.0456,0.0132,0.0521,2,0
A_W,0.0002,0.0063,0.0083,0.0249,-0.7686,0.0063,0.0149,2,0
A_MA,0.0078,0.2485,0.3258,0.9761,0.5126,0.2485,0.5820,2,0
P_EP,-0.0005,-0.0151,0.0076,-1.4876,0.0031,-0.0151,0.0040,2,-2
P_Pho,0.0003,0.0091,0.0313,-0.9532,0.0137,0.0091,0.0155,2,0
P_MP,0.0006,0.0191,0.0656,0.9604,0.0288,0.0191,0.0325,2,0
Eps_AB,-0.0012,-0.0249,-0.0041,0.0512,0.0196,-0.0249,-0.0694,-2,-2
Eps_EEps,-0.0212,-0.3645,0.0130,0.0600,-0.0366,-0.3645,0.0480,0,-2


In [30]:
num_variables = len(sm.prod_change_dict)
combinations = generate_combinations(num_variables, discrete_vals=[0, 1, 2])
print(len(combinations))
for c in tqdm(combinations, total=len(combinations)):
    dc = df * np.array(c).reshape(17, 1)

    #

129140163


  1%|▏         | 1619406/129140163 [01:20<1:45:36, 20124.09it/s]


## Solve with CO2 tax and biofuel policy

In [6]:
df_biofuel_quantities, df_biofuel_prices = sm.gen_results(
    robust_check=False, biofuel_tax=1
)
df_biofuel_quantities

{} {'self': <web_model.model.SolveModel object at 0x000001AD17240548>, 'sigma_U': 0.5, 'sigma_F': 1.23, 'sigma_nF': 1.8, 'sigma_A': 1.14, 'sigma_P': 0.2, 'sigma_nLA': 0.5, 'sigma_Eps': 1.8, 'sigma_Fi': 0.2, 'sigma_T': 0.2, 'sigma_Y': 0.5, 'Lambda_R': 0.37037037037037035, 'Lambda_E': 1, 'Lambda_W': 0.5586592178770949, 'Lambda_Pho': 0.6666666666666666, 'Lambda_M': 1, 'Q_LA': 0.53, 'Q_LT': 0.02, 'Q_AB': 0.038, 'Q_EpsA': 0.05, 'Q_EP': 0.014, 'Q_EFi': 0.004, 'GammaU_F': 0.1235, 'GammaF_Fi': 0.034, 'GammanF_Y': 0.991, 'GammanF_LU': 0.01711, 'GammaA_LA': 0.192, 'GammanLA_P': 0.0796, 'GammanLA_W': 0.0239, 'GammaP_EP': 0.1095, 'GammaEps_AB': 0.0037, 'GammaEps_EEps': 0.9433, 'GammaFi_EFi': 0.228, 'GammaT_LT': 0.3748, 'GammaY_EpsY': 0.0638, 'GammanLA_EpsA': 0.0412, 'GammaP_Pho': 0.3127, 'V_T': 0.05, 'V_A': 0.05, 'tau_E': 0.0, 'Lambda_MA': 1, 'Lambda_MT': 1, 'Lambda_MY': 1, 'Lambda_MP': 1, 'Lambda_MFi': 1, 'Q_LU': 0.44999999999999996, 'Q_EEps': 0.982, 'Q_AF': 0.962, 'Q_EpsY': 0.95, 'GammaU_nF': 0.

,description,max value,min value,mean value
0,$L_A$ $\text{(land-share agriculture)}$,0.0000,0.0000,0.0000
1,$L_T$ $\text{(land-share timber)}$,0.0000,0.0000,0.0000
2,$L_U$ $\text{(land-share other)}$,0.0000,0.0000,0.0000
3,$E$ $\text{(fossil-fuel extracted)}$,0.0000,0.0000,0.0000
4,$E_{\mathcal{E}}$ $\text{(fossil-fuel use ener...,0.0000,0.0000,0.0000
5,$E_P$ $\text{(fossil-fuel use fertilizer prod.)}$,0.0000,0.0000,0.0000
6,$E_F$ $\text{(fossil-fuel use fisheries)}$,0.0000,0.0000,0.0000
7,$A\;\;$ $\text{(agriculture total production)}$,0.0000,0.0000,0.0000
8,$A_B$ $\text{(agriculture prod. for biofuels)}$,0.0000,0.0000,0.0000
9,$A_F$ $\text{(agriculture prod. for food)}$,0.0000,0.0000,0.0000


# For PNAS Latex Table and Figure

In [7]:
df_full = (
    df_carbontax_quantities.copy()
    .rename(columns={"mean value": "outcome"})
    .loc[:, ["description", "outcome"]]
)
df_price_full = df_carbontax_prices.rename(columns={"mean value": "prices"}).loc[
    :, ["description", "prices"]
]

df_full["outcome_biofuel"] = df_biofuel_quantities.reset_index().rename(
    columns={"mean value": "outcome_biofuel"}
)["outcome_biofuel"]
df_price_full["prices_biofuel"] = df_biofuel_prices["mean value"]

df_full = df_full.rename(columns={"description": "index"})
df_price_full = df_price_full.rename(columns={"description": "index"})

df_full

,index,outcome,outcome_biofuel
0,$L_A$ $\text{(land-share agriculture)}$,0.0000,0.0000
1,$L_T$ $\text{(land-share timber)}$,0.0000,0.0000
2,$L_U$ $\text{(land-share other)}$,0.0000,0.0000
3,$E$ $\text{(fossil-fuel extracted)}$,0.0000,0.0000
4,$E_{\mathcal{E}}$ $\text{(fossil-fuel use ener...,0.0000,0.0000
5,$E_P$ $\text{(fossil-fuel use fertilizer prod.)}$,0.0000,0.0000
6,$E_F$ $\text{(fossil-fuel use fisheries)}$,0.0000,0.0000
7,$A\;\;$ $\text{(agriculture total production)}$,0.0000,0.0000
8,$A_B$ $\text{(agriculture prod. for biofuels)}$,0.0000,0.0000
9,$A_F$ $\text{(agriculture prod. for food)}$,0.0000,0.0000


### Generate model variables result table

In [9]:
from web_model import pb_table_generators as pt

t = pt.model_variable_result_table(
    params.var_desc_dict, params.price_desc_dict, df_full, df_price_full
)

KeyError: 'outcome_biofuel'

In [24]:
# import necessary packages
import numpy as np
import pandas as pd
import re
import os, sys
from itertools import product
from web_model import params, model  # Load parameters and model implementation.
import importlib

importlib.reload(params)
importlib.reload(model)
pd.options.display.float_format = "{:20,.4f}".format
print(pd.__version__)
print(np.__version__)
# Note: Code has been tested with with Pandas version 0.23.4 and Numpy version 1.15.1

0.25.1
1.16.5


In [25]:
print(params.param_dict["tau_E"])

0.4


In [9]:
param_list = list(params.df_typing_formatting.T.to_dict().values())
model_params = {row["keys"]: row["values"] for row in param_list}
sm = model.SolveModel(model_params)

df_carbontax_quantities, df_carbontax_prices = sm.gen_results(
    robust_check=False, biofuel_tax=0
)
df_carbontax_quantities
sm.pb_effects

df_full = (
    df_carbontax_quantities.copy()
    .rename(columns={"mean value": "outcome"})
    .loc[:, ["description", "outcome"]]
)
df_price_full = df_carbontax_prices.rename(columns={"mean value": "prices"}).loc[
    :, ["description", "prices"]
]

df_full = df_full.rename(columns={"description": "index"})
df_price_full = df_price_full.rename(columns={"description": "index"})

{} {'self': <web_model.model.SolveModel object at 0x000001B6AF673F88>, 'sigma_U': 0.5, 'sigma_F': 1.23, 'sigma_nF': 1.8, 'sigma_A': 1.14, 'sigma_P': 0.2, 'sigma_nLA': 0.5, 'sigma_Eps': 1.8, 'sigma_Fi': 0.2, 'sigma_T': 0.2, 'sigma_Y': 0.5, 'Lambda_R': 0.37037037037037035, 'Lambda_E': 1, 'Lambda_W': 0.5586592178770949, 'Lambda_Pho': 0.6666666666666666, 'Lambda_M': 1, 'Q_LA': 0.53, 'Q_LT': 0.02, 'Q_AB': 0.038, 'Q_EpsA': 0.05, 'Q_EP': 0.014, 'Q_EFi': 0.004, 'GammaU_F': 0.1235, 'GammaF_Fi': 0.034, 'GammanF_Y': 0.991, 'GammanF_LU': 0.01711, 'GammaA_LA': 0.192, 'GammanLA_P': 0.0796, 'GammanLA_W': 0.0239, 'GammaP_EP': 0.1095, 'GammaEps_AB': 0.0037, 'GammaEps_EEps': 0.9433, 'GammaFi_EFi': 0.228, 'GammaT_LT': 0.3748, 'GammaY_EpsY': 0.0638, 'GammanLA_EpsA': 0.0412, 'GammaP_Pho': 0.3127, 'V_T': 0.05, 'V_A': 0.05, 'tau_E': 0.0, 'Lambda_MA': 1, 'Lambda_MT': 1, 'Lambda_MY': 1, 'Lambda_MP': 1, 'Lambda_MFi': 1, 'Q_LU': 0.44999999999999996, 'Q_EEps': 0.982, 'Q_AF': 0.962, 'Q_EpsY': 0.95, 'GammaU_nF': 0.

# PB impacts
Set policy outcome dataframe from which to generate a planetary boundary impact analysis.

In [10]:
df_base_policy = df_carbontax_quantities.rename(
    columns={"mean value": "outcome", "description": "index"}
)[["index", "outcome"]].apply(
    lambda x: pd.Series(
        [
            re.search(r"\((.*?)\)", x["index"]).group(1).title(),
            re.findall("\$(.*?)\$", x["index"])[0].replace("\;", ""),
            x["outcome"],
        ],
        index=["index", "variable", "outcome"],
    ),
    axis=1,
)

df_base_policy

,index,variable,outcome
0,Land-Share Agriculture,L_A,-0.0226
1,Land-Share Timber,L_T,0.0138
2,Land-Share Other,L_U,0.0260
3,Fossil-Fuel Extracted,E,0.0057
4,Fossil-Fuel Use Energy Serv.,E_{\mathcal{E}},0.0115
5,Fossil-Fuel Use Fertilizer Prod.,E_P,-0.3938
6,Fossil-Fuel Use Fisheries,E_F,-0.0222
7,Agriculture Total Production,A,0.0528
8,Agriculture Prod. For Biofuels,A_B,0.1431
9,Agriculture Prod. For Food,A_F,0.0493


In [11]:
# Climate Impacts

# 44.153 GtCO2 in 2005
# https://www.ipcc.ch/pdf/assessment-report/ar5/wg3/ipcc_wg3_ar5_technical-summary.pdf
total_emissions = 44.153
df_base_policy.loc[:, "CO2 effect"] = 0
df_base_policy.loc[2, "CO2 effect"] = (
    -1 * 5.387 * df_base_policy.loc[2, "outcome"] / total_emissions
)  # change in natural land
df_base_policy.loc[3, "CO2 effect"] = (
    3.98 * df_base_policy.loc[3, "outcome"] / total_emissions
)  # fossil fuel extraction
df_base_policy.loc[4, "CO2 effect"] = (
    25.960 * df_base_policy.loc[4, "outcome"] / total_emissions
)  # Fossil-Fuel Use Energy Serv.
df_base_policy.loc[5, "CO2 effect"] = (
    0.575 * df_base_policy.loc[5, "outcome"] / total_emissions
)  # Fossils in fertilizer production
df_base_policy.loc[6, "CO2 effect"] = (
    0.14 * df_base_policy.loc[6, "outcome"] / total_emissions
)  # Fossils in fisheries
df_base_policy.loc[7, "CO2 effect"] = (
    6.093 * df_base_policy.loc[7, "outcome"] / total_emissions
)  # Emissions from Agriculture
df_base_policy.loc[19, "CO2 effect"] = (
    1.9 * df_base_policy.loc[19, "outcome"] / total_emissions
)  # Fossil-Fuel Use Final goods
total_co2_effect = round(sum(df_base_policy["CO2 effect"]), 4)

# Biodiversity impacts

total_num_threats = 25779  # Non-mutually exclusive sum
df_base_policy.loc[:, "biodiv-val"] = 0
df_base_policy.loc[3, "biodiv-val"] = 56  # Energy production (OIL and GAS)
df_base_policy.loc[16, "biodiv-val"] = 56  # Energy production (Renewable Energy)
df_base_policy.loc[17, "biodiv-val"] = 1118  # Over exploitation (Fishing)
df_base_policy.loc[18, "biodiv-val"] = 4049  # Over exploitation (Logging)
df_base_policy.loc[7, "biodiv-val"] = 5407 - 112  # Agricultural activity
df_base_policy.loc[13, "biodiv-val"] = 1523  # Pollution (Agriculture)
df_base_policy.loc[19, "biodiv-val"] = (
    907 + (1901 - 1523) + 236 + 1219 + 833
)  # Urban dev. (industrial) + Pollution (except Agriculture) + Human dist. (work) + Transport + Energy production (Mining)
df_base_policy.loc[:, "Biodiv. effect"] = (
    df_base_policy["outcome"] * df_base_policy["biodiv-val"] / total_num_threats
)

# Biogeochemical impacts

df_base_policy.loc[:, "Biogeochem. effect"] = 0
df_base_policy.loc[5, "Biogeochem. effect"] = df_base_policy.loc[5, "outcome"]
df_base_policy.loc[15, "Biogeochem. effect"] = df_base_policy.loc[15, "outcome"]

# Water impacts

df_base_policy.loc[:, "Freshwater effect"] = 0
df_base_policy.loc[14, "Freshwater effect"] = df_base_policy.loc[14, "outcome"]

# Ocean acidification impacts

df_base_policy.loc[:, "Ocean acid. effect"] = 0
df_base_policy.loc[:, "Ocean acid. effect"] = df_base_policy.loc[:, "CO2 effect"]

# Land use impacts

df_base_policy.loc[:, "Land-use effect"] = 0
df_base_policy.loc[2, "Land-use effect"] = df_base_policy.loc[2, "outcome"]

print("Total effect:", round(df_base_policy["Land-use effect"].sum(), 5), "%")
print(
    "Total effect (mHa):",
    round(3507 * df_base_policy["Land-use effect"].sum() / 100, 4),
)

# Ozone impacts

df_base_policy.loc[:, "Ozone effect"] = 0
df_base_policy.loc[3, "Ozone effect"] = (
    -1 if df_base_policy.loc[3, "outcome"] < 0 else 1
)  # N02 fossil fuels
df_base_policy.loc[7, "Ozone effect"] = (
    -1 if df_base_policy.loc[7, "outcome"] < 0 else 1
)  #  NO2 from agric.
df_base_policy.loc[8, "Ozone effect"] = (
    -1 if df_base_policy.loc[8, "outcome"] < 0 else 1
)  #  NO2 from biofuel burning.
df_base_policy.loc[19, "Ozone effect"] = (
    -1 if df_base_policy.loc[19, "outcome"] < 0 else 1
)  # NO2 from industry.
df_base_policy

# Chemical pollution

df_base_policy.loc[:, "Chem. effect"] = 0
df_base_policy.loc[3, "Chem. effect"] = (
    -1 * int(df_base_policy.loc[3, "outcome"] < 0)
    + df_base_policy.loc[3, "Chem. effect"]
)  # fossil fuels
df_base_policy.loc[7, "Chem. effect"] = (
    -1 * int(df_base_policy.loc[7, "outcome"] < 0)
    + df_base_policy.loc[7, "Chem. effect"]
)  # agric.
df_base_policy.loc[19, "Chem. effect"] = (
    -1 * int(df_base_policy.loc[19, "outcome"] < 0)
    + df_base_policy.loc[19, "Chem. effect"]
)  # industry.

df_base_policy


# df_final_impact = df_base_policy[['index', 'variable', 'CO2 effect', 'Biodiv. effect', 'Biogeochem. effect', 'Freshwater effect',
# 'Ocean acid. effect', 'Land-use effect', 'Aerosol effect', 'Ozone effect', 'Chem. effect']]
# df_final_impact = df_base_policy[['index', 'variable', 'CO2 effect', 'Biodiv. effect', 'Biogeochem. effect', 'Freshwater effect',
#                                   'Ocean acid. effect', 'Land-use effect', 'Ozone effect', 'Chem. effect']]
df_final_impact.to_excel("PB_impacts.xlsx")
df_final_impact


import importlib
from web_model import pb_table_generators as pt

importlib.reload(pt)
pt.model_pbimpact_result_table(df_final_impact)

Total effect: 0.02603 %
Total effect (mHa): 0.9128
index                 Land-Share AgricultureLand-Share TimberLand-Sh...
variable              L_AL_TL_UEE_{\mathcal{E}}E_PE_FAA_BA_F\mathcal...
CO2 effect                                                       0.0066
Biodiv. effect                                                  -0.0068
Biogeochem. effect                                              -0.7402
Freshwater effect                                                0.0228
Ocean acid. effect                                               0.0066
Land-use effect                                                  0.0260
Ozone effect                                                          4
Chem. effect                                                          0
dtype: object


## Aersols

The impact of aerosols in terms of their effect on climate change can be estimated via radiative forcing (see table 8.4 in
https://www.ipcc.ch/site/assets/uploads/2018/02/WG1AR5_Chapter08_FINAL.pdf). As to the PB impact we adopt the aersol optical
depth measure used in Steffen et al. we base our estinates on data from 3 sources:
1) Streets et al:  Anthropogenic and natural contributions to regional trends inaerosol optical depth, 1980–2006: https://pdfs.semanticscholar.org/8dbf/77f8c55ce39c7654a78ebe129098054fc47b.pdf
2) Lamarque. Historical (1850–2000) gridded anthropogenic and biomass burningemissions http://pure.iiasa.ac.at/id/eprint/9279/1/acp-10-7017-2010.pdf
3) Levine. Biomass Burning: The Cycling of Gases and Particulates from the Biosphere to the Atmosphere https://www-sciencedirect-com.ezp.sub.su.se/science/article/pii/B0080437516041438

Spurious Notes: 

In interpretation of results note that radiative forcing and AOD (control variable in PB framework) are inversely related (see e.g. /http://www.aaqr.org/files/article/6789/5_AAQR-17-12-AC3-0600_38-48.pdf ; https://www.nist.gov/sites/default/files/documents/mml/opt_prop_schwartz.pdf ; http://cdn.intechopen.com/pdfs/38765/InTech-Aerosol_direct_radiative_forcing_a_review.pdf)

 From https://www.bp.com/content/dam/bp/business-sites/en/global/corporate/pdfs/energy-economics/statistical-review/bp-stats-review-2019-full-report.pdf we have that: Biofuels: 95371 Thousand tonnes oil equivalent Fossilfuels (Oil, gas coal): 4662.1 + 3309.4 + 3772.1 Million tonnes oil equivalent.


### From Streets et al we can extract global AOD average measures.

In [13]:
Streets_table2 = pd.read_csv(
    "Streets_table2.csv", delimiter=";", index_col="Region"
)  # Emissions, Total Mass Burden, and AOD by Region for 2001
Streets_table4 = pd.read_csv(
    "Streets_table4.csv", delimiter=";", index_col="Region"
)  # Average Contributions of Aerosol Types to Estimated AOD From 1980 to 2006
Streets_table2 = Streets_table2.loc[Streets_table2.Parameter == "AOD"]
# df_aerosols = pd.read_csv('Streets_table2.txt', sep='\t', lineterminator='\r')
df_aerosols = Streets_table2.copy()
for idx, row in df_aerosols.iterrows():
    df_aerosols.at[idx, "Sulfur"] = (
        row["Sulfur"] * Streets_table4.loc[idx, "Sulfur-Anthro"] / 100
    )
    df_aerosols.at[idx, "BC"] = row["BC"] * Streets_table4.loc[idx, "BC-Anthro"] / 100
    df_aerosols.at[idx, "OC"] = row["OC"] * Streets_table4.loc[idx, "OC-Anthro"] / 100

In [14]:
print(df_aerosols[["Sulfur", "BC", "OC"]].mean(axis=0))
df_aerosols

Sulfur                 0.0392
BC                     0.0003
OC                     0.0011
dtype: float64


,Parameter,Sulfur,BC,OC,Sea Salt,Dust
Region,,,,,,
United States,AOD,0.0466,0.0000,0.0003,0.0000,0.0300
South America,AOD,0.0092,0.0000,0.0007,0.0100,0.0100
OECD Europe,AOD,0.0509,0.0003,0.0003,0.0200,0.0500
Russia,AOD,0.1037,0.0003,0.0004,0.0100,0.0600
Southern Africa,AOD,0.0106,0.0003,0.0016,0.0100,0.0100
South Asia,AOD,0.0285,0.0006,0.0039,0.0100,0.0600
East Asia,AOD,0.0504,0.0004,0.0011,0.0100,0.0800
Southeast Asia,AOD,0.0139,0.0003,0.0008,0.0100,0.0100


### From: Lamarque et al.: 1850–2000 gridded anthropogenic and biomass burning emissions

In [15]:
Biomass_burning_emissions = {
    "BC": 2.61,
    "OC": 23.25,
    "SO2": 3.84,
}  # From table 9. in J.-F. Lamarque et al.: 1850–2000 gridded anthropogenic and biomass burning emissions
Global_anthropogenic_emissions = {
    "BC": 5.02,
    "OC": 12.56,
    "SO2": 92.71,
}  # From table 7 in  J.-F. Lamarque et al.: 1850–2000 gridded anthropogenic and biomass burning emissions

Global_anthropogenic_biomass_burning_emissions = {
    k: v * 0.9 for k, v in Biomass_burning_emissions.items()
}
# Note: 0.9 is a measure of percentage of biomass burning from land-use change in https://www-sciencedirect-com.ezp.sub.su.se/science/article/pii/B0080437516041438

Global_anthropogenic_incl_biomass_burning_emissions = {
    k: v + Global_anthropogenic_biomass_burning_emissions[k]
    for k, v in Global_anthropogenic_emissions.items()
}

Share_Global_anthropogenic_biomass_burning_emissions = {
    k: Global_anthropogenic_biomass_burning_emissions[k] / v
    for k, v in Global_anthropogenic_incl_biomass_burning_emissions.items()
}
sg = Share_Global_anthropogenic_biomass_burning_emissions
sum(list(sg.values()))

0.9796123431858057

### Compute global anthropogenic AOD 

In [17]:
df_aerosols_mean = df_aerosols[["Sulfur", "BC", "OC"]].mean(axis=0)
df_aerosols_mean_from_fossil_and_bio_fuels = df_aerosols_mean * np.array(
    [1 - sg["SO2"], 1 - sg["BC"], 1 - sg["OC"]]
)
df_aerosols_mean_from_biomass = (
    df_aerosols_mean - df_aerosols_mean_from_fossil_and_bio_fuels
)

AOD_global_fbf = df_aerosols_mean_from_fossil_and_bio_fuels.sum()
AOD_global_biomass = df_aerosols_mean_from_biomass.sum()
print(AOD_global_fbf)
print(AOD_global_biomass)

0.038414748418532686
0.0022015015814673216


In [18]:
Share_of_AOD_attributed_to_biofuels = AOD_global_biomass / (
    AOD_global_biomass + AOD_global_fbf
)
Share_of_AOD_attributed_to_biofuels

0.05420248254005038

### Compute Aerosol effect from policy

In [19]:
Q_EP = 0.014  # share of fossil fuel used for fertilizer prod.
Q_EFi = 0.004  # share of fossil fuel used for fisheries prod.
Q_EEps = 1 - Q_EFi - Q_EP
Q_bio = 95371 / (
    1000 * (4662.1 + 3309.4 + 3772.1) + 95371
)  # Share of biofuels in tot biofuel+fossil prod.
Q_ff = 1 - Q_bio

# 'Aerosol effect': the net change in radiative forcing from imposing a carbon tax (note: a positive value implies a increase in radiative forcing aka a decrease of AOD)
df_base_policy.loc[:, "Aerosol effect"] = 0
df_base_policy.loc[2, "Aerosol effect"] = (
    -AOD_global_biomass * df_base_policy.loc[2, "outcome"]
)  # Natural land
df_base_policy.loc[4, "Aerosol effect"] = (
    AOD_global_fbf * df_base_policy.loc[4, "outcome"] * Q_ff * Q_EEps
)  # Fossil fuel consump. energy services
df_base_policy.loc[5, "Aerosol effect"] = (
    AOD_global_fbf * df_base_policy.loc[5, "outcome"] * Q_ff * Q_EP
)  # Fossil fuel consump.  Fertilizer Prod
df_base_policy.loc[6, "Aerosol effect"] = (
    AOD_global_fbf * df_base_policy.loc[6, "outcome"] * Q_ff * Q_EFi
)  # Fossil fuel consump. fisheries
df_base_policy.loc[8, "Aerosol effect"] = (
    AOD_global_fbf * df_base_policy.loc[8, "outcome"] * Q_bio
)  # Biofuel fuel prod

print("Total effect:", round(df_base_policy["Aerosol effect"].sum(), 5), "%")
print(
    "Total effect (AOD):",
    (AOD_global_fbf + AOD_global_biomass)
    * df_base_policy["Aerosol effect"].sum()
    / 100,
)
df_base_policy

Total effect: 0.0002 %
Total effect (AOD): 8.261549937719845e-08


,index,variable,outcome,CO2 effect,biodiv-val,Biodiv. effect,Biogeochem. effect,Freshwater effect,Ocean acid. effect,Land-use effect,Ozone effect,Chem. effect,Aerosol effect
0,Land-Share Agriculture,L_A,-0.0226,0.0000,0,-0.0000,0.0000,0.0000,0.0000,0.0000,0,0,0.0000
1,Land-Share Timber,L_T,0.0138,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000,0,0,0.0000
2,Land-Share Other,L_U,0.0260,-0.0032,0,0.0000,0.0000,0.0000,-0.0032,0.0260,0,0,-0.0001
3,Fossil-Fuel Extracted,E,0.0057,0.0005,56,0.0000,0.0000,0.0000,0.0005,0.0000,1,0,0.0000
4,Fossil-Fuel Use Energy Serv.,E_{\mathcal{E}},0.0115,0.0068,0,0.0000,0.0000,0.0000,0.0068,0.0000,0,0,0.0004
5,Fossil-Fuel Use Fertilizer Prod.,E_P,-0.3938,-0.0051,0,-0.0000,-0.3938,0.0000,-0.0051,0.0000,0,0,-0.0002
6,Fossil-Fuel Use Fisheries,E_F,-0.0222,-0.0001,0,-0.0000,0.0000,0.0000,-0.0001,0.0000,0,0,-0.0000
7,Agriculture Total Production,A,0.0528,0.0073,5295,0.0109,0.0000,0.0000,0.0073,0.0000,1,0,0.0000
8,Agriculture Prod. For Biofuels,A_B,0.1431,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000,1,0,0.0000
9,Agriculture Prod. For Food,A_F,0.0493,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000,0,0,0.0000


## Climate impacts
### Methodology

In order to calculate the climate impact from the change in model variables resulting from the carbon tax we use data from two sources http://www.wri.org/sites/default/files/world_ghg_flow_chart_2005.png and https://www.nature.com/news/one-third-of-our-greenhouse-gas-emissions-come-from-agriculture-1.11708. From the graph of sectorial CO2 emissions from WRI we make the following assumptions in terms of how this connects to our model varaibles. 

* From the energy related emissions in the WRI graph (a total 66.5% = 29,36 GtCO2 eq). We wish to split this total into emissions from energy service production, fossil fuel extraction and emissions from fertilizer production. From the WRI graph we know that 6.4%+1.3%+2.2%=9.9% (3.98 GtCO2 eq) the total energy related emissions is due to extraction processes. From the Nature article we have that 0.575 GtCO2 eq is estimated to be released from fertilizer production. Hence we split the total energy related emission of 29,36 GtCO2 eq as follows
  * 24,41 GtCO2 eq are connected to the energy services variable in our model. (55.3%)
  * 3.98 GtCO2 eq are connected to the fossil fuel extraction variable in our model. (9.9%)
  * 0.575 GtCO2 eq are connected to fertilizer production. (1.3%)
* Emissions from industrial processes are linked to Final goods production in our model (In total 4.3% = 1.9 GtCO2 eq)
* Emission from land-use change are linked to the change in natural land in our model (12.2% = 5.387 GtCO2 eq).
* Emissions from agriculture are linked to total agricultural production in our model (13.8% = 6.093 GtCO2 eq)
* Emissions from fisheries in 2005 where approximately 0.14 GtCO2 (0.32%) (see https://s3-us-west-2.amazonaws.com/legacy.seaaroundus/researcher/dpauly/PDF/2019/Journal+Articles/Greer+et+al%2C+2019%2C+CO2.pdf)

Totals 97.4 % of total emissions.

In [9]:
# 44.153 GtCO2 in 2005
# https://www.ipcc.ch/pdf/assessment-report/ar5/wg3/ipcc_wg3_ar5_technical-summary.pdf
total_emissions = 44.153
df_base_policy.loc[:, "CO2 effect"] = 0
df_base_policy.loc[2, "CO2 effect"] = (
    -1 * 5.387 * df_base_policy.loc[2, "outcome"] / total_emissions
)  # change in natural land
df_base_policy.loc[3, "CO2 effect"] = (
    3.98 * df_base_policy.loc[3, "outcome"] / total_emissions
)  # fossil fuel extraction
df_base_policy.loc[4, "CO2 effect"] = (
    25.960 * df_base_policy.loc[4, "outcome"] / total_emissions
)  # Fossil-Fuel Use Energy Serv.
df_base_policy.loc[5, "CO2 effect"] = (
    0.575 * df_base_policy.loc[5, "outcome"] / total_emissions
)  # Fossils in fertilizer production
df_base_policy.loc[6, "CO2 effect"] = (
    0.14 * df_base_policy.loc[6, "outcome"] / total_emissions
)  # Fossils in fisheries
df_base_policy.loc[7, "CO2 effect"] = (
    6.093 * df_base_policy.loc[7, "outcome"] / total_emissions
)  # Emissions from Agriculture
df_base_policy.loc[19, "CO2 effect"] = (
    1.9 * df_base_policy.loc[19, "outcome"] / total_emissions
)  # Fossil-Fuel Use Final goods
# df_base_policy.loc[:, 'CO2 effect'] = df_base_policy.loc[:, 'CO2 effect']/100
total_co2_effect = round(sum(df_base_policy["CO2 effect"]), 4)

print("Total CO2 effect percent: ", total_co2_effect, " % of total")
print("Total CO2 GtCO2: ", total_co2_effect * total_emissions / 100, " GtCO2")

df_base_policy

Total CO2 effect percent:  0.0  % of total
Total CO2 GtCO2:  0.0  GtCO2


,index,variable,outcome,CO2 effect
0,Land-Share Agriculture,L_A,0.0000,0.0000
1,Land-Share Timber,L_T,0.0000,0.0000
2,Land-Share Other,L_U,0.0000,-0.0000
3,Fossil-Fuel Extracted,E,0.0000,0.0000
4,Fossil-Fuel Use Energy Serv.,E_{\mathcal{E}},0.0000,0.0000
5,Fossil-Fuel Use Fertilizer Prod.,E_P,0.0000,0.0000
6,Fossil-Fuel Use Fisheries,E_F,0.0000,0.0000
7,Agriculture Total Production,A,0.0000,0.0000
8,Agriculture Prod. For Biofuels,A_B,0.0000,0.0000
9,Agriculture Prod. For Food,A_F,0.0000,0.0000


## Biodiversity impacts

For biodiversity impacts we use the results from the Nature article https://www.nature.com/news/biodiversity-the-ravages-of-guns-nets-and-bulldozers-1.20381. 

Here we have connected the categories of threatened species by driver in the article to our model variables and multiplied the numbers in the article by the percentage change in our model variables.

In [10]:
total_num_threats = 25779  # Non-mutually exclusive sum

df_base_policy.loc[:, "biodiv-val"] = 0
df_base_policy.loc[3, "biodiv-val"] = 56  # Energy production (OIL and GAS)
df_base_policy.loc[16, "biodiv-val"] = 56  # Energy production (Renewable Energy)
df_base_policy.loc[17, "biodiv-val"] = 1118  # Over exploitation (Fishing)
df_base_policy.loc[18, "biodiv-val"] = 4049  # Over exploitation (Logging)
df_base_policy.loc[7, "biodiv-val"] = 5407 - 112  # Agricultural activity
df_base_policy.loc[13, "biodiv-val"] = 1523  # Pollution (Agriculture)
df_base_policy.loc[19, "biodiv-val"] = (
    907 + (1901 - 1523) + 236 + 1219 + 833
)  # Urban dev. (industrial) + Pollution (except Agriculture) + Human dist. (work) + Transport + Energy production (Mining)
df_base_policy.loc[:, "Biodiv. effect"] = (
    df_base_policy["outcome"] * df_base_policy["biodiv-val"] / total_num_threats
)
df_base_policy

,index,variable,outcome,CO2 effect,biodiv-val,Biodiv. effect
0,Land-Share Agriculture,L_A,0.0000,0.0000,0,0.0000
1,Land-Share Timber,L_T,0.0000,0.0000,0,0.0000
2,Land-Share Other,L_U,0.0000,-0.0000,0,0.0000
3,Fossil-Fuel Extracted,E,0.0000,0.0000,56,0.0000
4,Fossil-Fuel Use Energy Serv.,E_{\mathcal{E}},0.0000,0.0000,0,0.0000
5,Fossil-Fuel Use Fertilizer Prod.,E_P,0.0000,0.0000,0,0.0000
6,Fossil-Fuel Use Fisheries,E_F,0.0000,0.0000,0,0.0000
7,Agriculture Total Production,A,0.0000,0.0000,5295,0.0000
8,Agriculture Prod. For Biofuels,A_B,0.0000,0.0000,0,0.0000
9,Agriculture Prod. For Food,A_F,0.0000,0.0000,0,0.0000


## Biogeochemical 

We assume nitrogen flows are proportional to fossil fuel use. Hence a percentage increase in fossil fuel use in fertilizer production is translated to the same percentage increase in the nitrogen flow. Concerning phosphate this related one-to-one with phosphate extraction in our model. The results are: 

In [11]:
df_base_policy.loc[:, "Biogeochem. effect"] = 0
df_base_policy.loc[5, "Biogeochem. effect"] = df_base_policy.loc[5, "outcome"]
df_base_policy.loc[15, "Biogeochem. effect"] = df_base_policy.loc[15, "outcome"]

print(
    "Total Phosp. effect percent: ",
    round(df_base_policy.loc[15, "Biogeochem. effect"], 4),
    "%",
)
print(
    "Total Nitrog. effect percent: ",
    round(df_base_policy.loc[5, "Biogeochem. effect"], 4),
    " %",
)

print(
    "Total Phosp. effect (Gg P y-1): ",
    round(14000 * df_base_policy.loc[15, "Biogeochem. effect"] / 100, 4),
    "Gg P y-1",
)
print(
    "Total Nitrog. effect (Tg P y-1): ",
    round(150 * df_base_policy.loc[5, "Biogeochem. effect"] / 100, 4),
    "Tg P y-1",
)

df_base_policy

Total Phosp. effect percent:  0.0 %
Total Nitrog. effect percent:  0.0  %
Total Phosp. effect (Gg P y-1):  0.0 Gg P y-1
Total Nitrog. effect (Tg P y-1):  0.0 Tg P y-1


,index,variable,outcome,CO2 effect,biodiv-val,Biodiv. effect,Biogeochem. effect
0,Land-Share Agriculture,L_A,0.0000,0.0000,0,0.0000,0.0000
1,Land-Share Timber,L_T,0.0000,0.0000,0,0.0000,0.0000
2,Land-Share Other,L_U,0.0000,-0.0000,0,0.0000,0.0000
3,Fossil-Fuel Extracted,E,0.0000,0.0000,56,0.0000,0.0000
4,Fossil-Fuel Use Energy Serv.,E_{\mathcal{E}},0.0000,0.0000,0,0.0000,0.0000
5,Fossil-Fuel Use Fertilizer Prod.,E_P,0.0000,0.0000,0,0.0000,0.0000
6,Fossil-Fuel Use Fisheries,E_F,0.0000,0.0000,0,0.0000,0.0000
7,Agriculture Total Production,A,0.0000,0.0000,5295,0.0000,0.0000
8,Agriculture Prod. For Biofuels,A_B,0.0000,0.0000,0,0.0000,0.0000
9,Agriculture Prod. For Food,A_F,0.0000,0.0000,0,0.0000,0.0000


## Water

In [12]:
df_base_policy.loc[:, "Freshwater effect"] = 0
df_base_policy.loc[14, "Freshwater effect"] = df_base_policy.loc[14, "outcome"]

print(
    "Total Water effect percent: ",
    round(df_base_policy.loc[14, "Freshwater effect"], 4),
    "%",
)
print(
    "Total Water. effect (km3 yr-1): ",
    round(2600 * df_base_policy.loc[14, "Freshwater effect"] / 100, 4),
    "km3 yr-1",
)

df_base_policy

Total Water effect percent:  0.0 %
Total Water. effect (km3 yr-1):  0.0 km3 yr-1


,index,variable,outcome,CO2 effect,biodiv-val,Biodiv. effect,Biogeochem. effect,Freshwater effect
0,Land-Share Agriculture,L_A,0.0000,0.0000,0,0.0000,0.0000,0.0000
1,Land-Share Timber,L_T,0.0000,0.0000,0,0.0000,0.0000,0.0000
2,Land-Share Other,L_U,0.0000,-0.0000,0,0.0000,0.0000,0.0000
3,Fossil-Fuel Extracted,E,0.0000,0.0000,56,0.0000,0.0000,0.0000
4,Fossil-Fuel Use Energy Serv.,E_{\mathcal{E}},0.0000,0.0000,0,0.0000,0.0000,0.0000
5,Fossil-Fuel Use Fertilizer Prod.,E_P,0.0000,0.0000,0,0.0000,0.0000,0.0000
6,Fossil-Fuel Use Fisheries,E_F,0.0000,0.0000,0,0.0000,0.0000,0.0000
7,Agriculture Total Production,A,0.0000,0.0000,5295,0.0000,0.0000,0.0000
8,Agriculture Prod. For Biofuels,A_B,0.0000,0.0000,0,0.0000,0.0000,0.0000
9,Agriculture Prod. For Food,A_F,0.0000,0.0000,0,0.0000,0.0000,0.0000


## Ocean acidification

In [13]:
df_base_policy.loc[:, "Ocean acid. effect"] = 0
df_base_policy.loc[:, "Ocean acid. effect"] = df_base_policy.loc[:, "CO2 effect"]
df_base_policy

,index,variable,outcome,CO2 effect,biodiv-val,Biodiv. effect,Biogeochem. effect,Freshwater effect,Ocean acid. effect
0,Land-Share Agriculture,L_A,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000
1,Land-Share Timber,L_T,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000
2,Land-Share Other,L_U,0.0000,-0.0000,0,0.0000,0.0000,0.0000,-0.0000
3,Fossil-Fuel Extracted,E,0.0000,0.0000,56,0.0000,0.0000,0.0000,0.0000
4,Fossil-Fuel Use Energy Serv.,E_{\mathcal{E}},0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000
5,Fossil-Fuel Use Fertilizer Prod.,E_P,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000
6,Fossil-Fuel Use Fisheries,E_F,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000
7,Agriculture Total Production,A,0.0000,0.0000,5295,0.0000,0.0000,0.0000,0.0000
8,Agriculture Prod. For Biofuels,A_B,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000
9,Agriculture Prod. For Food,A_F,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000


## Land-use

In [14]:
df_base_policy.loc[:, "Land-use effect"] = 0
df_base_policy.loc[2, "Land-use effect"] = df_base_policy.loc[2, "outcome"]

print("Total effect:", round(df_base_policy["Land-use effect"].sum(), 5), "%")
print(
    "Total effect (mHa):",
    round(3507 * df_base_policy["Land-use effect"].sum() / 100, 4),
)

df_base_policy

Total effect: 0.0 %
Total effect (mHa): 0.0


,index,variable,outcome,CO2 effect,biodiv-val,Biodiv. effect,Biogeochem. effect,Freshwater effect,Ocean acid. effect,Land-use effect
0,Land-Share Agriculture,L_A,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000
1,Land-Share Timber,L_T,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000
2,Land-Share Other,L_U,0.0000,-0.0000,0,0.0000,0.0000,0.0000,-0.0000,0.0000
3,Fossil-Fuel Extracted,E,0.0000,0.0000,56,0.0000,0.0000,0.0000,0.0000,0.0000
4,Fossil-Fuel Use Energy Serv.,E_{\mathcal{E}},0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000
5,Fossil-Fuel Use Fertilizer Prod.,E_P,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000
6,Fossil-Fuel Use Fisheries,E_F,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000
7,Agriculture Total Production,A,0.0000,0.0000,5295,0.0000,0.0000,0.0000,0.0000,0.0000
8,Agriculture Prod. For Biofuels,A_B,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000
9,Agriculture Prod. For Food,A_F,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000


## Stratospheric Ozone
Anthropogenic ozone-depleting substances are essentially under control via the montreal protocol. Other than that
ozone depletion is positively correlated with climate change and NO2. The two later are largely due to agriculture and fossil fuel use and industry.

We do not consider cross-boundary effects and thus disregard from climate change.

Ozone is impacted by fossil fuel burning (energy services and fisheries), agricultural production (released e.g. from soil), burning of biofuels and industry.

We don´t quantify this process but rather assume a qualitative direction based on the above. If all affected variables become negative then the overall qualitative prediction will be negative.

In [15]:
df_base_policy.loc[:, "Ozone effect"] = 0
df_base_policy.loc[3, "Ozone effect"] = (
    -1 if df_base_policy.loc[3, "outcome"] < 0 else 1
)  # N02 fossil fuels
df_base_policy.loc[7, "Ozone effect"] = (
    -1 if df_base_policy.loc[7, "outcome"] < 0 else 1
)  #  NO2 from agric.
df_base_policy.loc[8, "Ozone effect"] = (
    -1 if df_base_policy.loc[8, "outcome"] < 0 else 1
)  #  NO2 from biofuel burning.
df_base_policy.loc[19, "Ozone effect"] = (
    -1 if df_base_policy.loc[19, "outcome"] < 0 else 1
)  # NO2 from industry.

df_base_policy

,index,variable,outcome,CO2 effect,biodiv-val,Biodiv. effect,Biogeochem. effect,Freshwater effect,Ocean acid. effect,Land-use effect,Ozone effect
0,Land-Share Agriculture,L_A,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000,0
1,Land-Share Timber,L_T,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000,0
2,Land-Share Other,L_U,0.0000,-0.0000,0,0.0000,0.0000,0.0000,-0.0000,0.0000,0
3,Fossil-Fuel Extracted,E,0.0000,0.0000,56,0.0000,0.0000,0.0000,0.0000,0.0000,1
4,Fossil-Fuel Use Energy Serv.,E_{\mathcal{E}},0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000,0
5,Fossil-Fuel Use Fertilizer Prod.,E_P,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000,0
6,Fossil-Fuel Use Fisheries,E_F,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000,0
7,Agriculture Total Production,A,0.0000,0.0000,5295,0.0000,0.0000,0.0000,0.0000,0.0000,1
8,Agriculture Prod. For Biofuels,A_B,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000,1
9,Agriculture Prod. For Food,A_F,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000,0


# Chemical pollution

Inline with Stratospheric ozone we do a qualitative assesment on this process based on model variables fossil fuel extracted, agriculture total and final goods.


In [16]:
df_base_policy.loc[:, "Chem. effect"] = 0
df_base_policy.loc[3, "Chem. effect"] = (
    -1 * int(df_base_policy.loc[3, "outcome"] < 0)
    + df_base_policy.loc[3, "Chem. effect"]
)  # fossil fuels
df_base_policy.loc[7, "Chem. effect"] = (
    -1 * int(df_base_policy.loc[7, "outcome"] < 0)
    + df_base_policy.loc[7, "Chem. effect"]
)  # agric.
df_base_policy.loc[19, "Chem. effect"] = (
    -1 * int(df_base_policy.loc[19, "outcome"] < 0)
    + df_base_policy.loc[19, "Chem. effect"]
)  # industry.


df_base_policy

,index,variable,outcome,CO2 effect,biodiv-val,Biodiv. effect,Biogeochem. effect,Freshwater effect,Ocean acid. effect,Land-use effect,Ozone effect,Chem. effect
0,Land-Share Agriculture,L_A,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000,0,0
1,Land-Share Timber,L_T,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000,0,0
2,Land-Share Other,L_U,0.0000,-0.0000,0,0.0000,0.0000,0.0000,-0.0000,0.0000,0,0
3,Fossil-Fuel Extracted,E,0.0000,0.0000,56,0.0000,0.0000,0.0000,0.0000,0.0000,1,0
4,Fossil-Fuel Use Energy Serv.,E_{\mathcal{E}},0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000,0,0
5,Fossil-Fuel Use Fertilizer Prod.,E_P,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000,0,0
6,Fossil-Fuel Use Fisheries,E_F,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000,0,0
7,Agriculture Total Production,A,0.0000,0.0000,5295,0.0000,0.0000,0.0000,0.0000,0.0000,1,0
8,Agriculture Prod. For Biofuels,A_B,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000,1,0
9,Agriculture Prod. For Food,A_F,0.0000,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000,0,0


## Final output

In [17]:
# df_final_impact = df_base_policy[['index', 'variable', 'CO2 effect', 'Biodiv. effect', 'Biogeochem. effect', 'Freshwater effect',
# 'Ocean acid. effect', 'Land-use effect', 'Aerosol effect', 'Ozone effect', 'Chem. effect']]
df_final_impact = df_base_policy[
    [
        "index",
        "variable",
        "CO2 effect",
        "Biodiv. effect",
        "Biogeochem. effect",
        "Freshwater effect",
        "Ocean acid. effect",
        "Land-use effect",
        "Ozone effect",
        "Chem. effect",
    ]
]
df_final_impact.to_excel("PB_impacts.xlsx")
df_final_impact

,index,variable,CO2 effect,Biodiv. effect,Biogeochem. effect,Freshwater effect,Ocean acid. effect,Land-use effect,Ozone effect,Chem. effect
0,Land-Share Agriculture,L_A,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0
1,Land-Share Timber,L_T,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0
2,Land-Share Other,L_U,-0.0000,0.0000,0.0000,0.0000,-0.0000,0.0000,0,0
3,Fossil-Fuel Extracted,E,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1,0
4,Fossil-Fuel Use Energy Serv.,E_{\mathcal{E}},0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0
5,Fossil-Fuel Use Fertilizer Prod.,E_P,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0
6,Fossil-Fuel Use Fisheries,E_F,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0
7,Agriculture Total Production,A,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1,0
8,Agriculture Prod. For Biofuels,A_B,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1,0
9,Agriculture Prod. For Food,A_F,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0


### Generate pb impact tables

In [18]:
import importlib
from web_model import pb_table_generators as pt

importlib.reload(pt)
pt.model_pbimpact_result_table(df_final_impact)

\begin{tabular}{llllllllll} 
  & \textit{Climate} & \textit{Biodiv.} & \textit{Biochem.} & \textit{Freshw.} & \textit{Ocean acid.} & \textit{Land-use}  & \textit{Aerosols} & \textit{Ozone} & \textit{Chem.} \\ 
\hline
\hline
Total impact & 0.0 & 0.0 & 0.0 & 0.0 & 0.0 & 0.0 & 4 & -1 & 1 \\ 
\hline 
\end{tabular} 


### Generate parameter tables

In [239]:
print(pt.param_expshare_table())
print(pt.param_qshare_table())
print(pt.param_elast_table())

25
\begin{table}[!ht]
\centering
\caption{\textbf{Parameters: expenditure shares (source: GTAP)}}
\begin{tabular}{ll} 
\hline
\textit{Expenditure share} & \textit{value} \\
\hline
$\prodel{A}{L_A}$ & 19.2\%  \\
$\prodel{\tilde{L}_A}{P}$ & 8.0\%   \\ 
$\prodel{\tilde{L}_A}{W}$ & 2.4\%  \\ 
$\prodel{\tilde{L}_A}{\mathcal{E}_A}$ & 4.1\%  \\ 
$\prodel{\mathcal{E}}{A_B}$ & 0.4\%   \\
$\prodel{\mathcal{E}}{E_\mathcal{E}}$ & 94.3\%   \\
$\prodel{U}{\mathcal{F}}$ & 12.3\%   \\ 
$\prodel{\mathcal{F}}{F}$ & 3.4\%   \\ 
$\prodel{\tilde{\mathcal{F}}}{Y}$ & 99.1\%   \\ 
$\prodel{\tilde{\mathcal{F}}}{L_U}$ & 1.7\%   \\ 
$\prodel{T}{L_T}$ & 37.5\%   \\
$\prodel{Y}{\mathcal{E}_Y}$ & 6.4\%   \\
$\prodel{P}{E_P}$ & 10.9\%   \\
$\prodel{P}{\mathcal{P}}$ & 31.3\%   \\
$\prodel{F}{E_F}$ & 22.8\%   \\
\hline
\end{tabular}
\label{Tab:FactorShares}
\end{table}
None
\begin{table}[!ht]
\centering
\caption{\textbf{Parameters - Quantity shares}}
\begin{tabular}{lll} 
\hline
\textit{Parameter} & \textit{source} & 

### Generate sensitivity table

In [246]:
pt.model_variable_sensitivity_table(
    var_desc_dict, df_carbontax_quantities, df_biofuel_quantities
)

\begin{tabular}{lrrrr}
\hline
& \multicolumn{2}{c}{}  & \multicolumn{2}{c}{\textit{Carbon Tax +}}  \\ 
& \multicolumn{2}{c}{\textit{Carbon tax}}  & \multicolumn{2}{c}{\textit{Biofuel policy} }  \\ 
\textit{Variable} & \textit{Min} & \textit{Max} & \textit{Min} & \textit{Max}  \\ 
\hline
\multicolumn{5}{l}{\textit{Agricultural Sector: Production}} \\ 
\hspace{0.2cm}Total & -0.0173 & 0.0303 & -0.0813 & -0.0229 \\ 
\hspace{0.2cm}Biofuels & 0.19 & 1.5561 & -1.6504 & -0.3552 \\ 
\hspace{0.2cm}Food & -0.06 & -0.0157 & -0.0395 & 0.0211 \\ 
\hspace{0.2cm}Total & -0.003 & 0.034 & -0.045 & -0.0 \\ 
\hspace{0.2cm}Biofuels & 0.717 & 0.034 & -1.008 & -0.0 \\ 
\hspace{0.2cm}Food & -0.032 & 0.034 & -0.007 & -0.0 \\ 
\hspace{0.2cm}Total & -0.0173 & 0.0303 & -0.0813 & -0.0229 \\ 
\hspace{0.2cm}Biofuels & 0.19 & 1.5561 & -1.6504 & -0.3552 \\ 
\hspace{0.2cm}Food & -0.06 & -0.0157 & -0.0395 & 0.0211 \\ 
\multicolumn{5}{l}{\textit{Agricultural Sector: Inputs}} \\ 
\hspace{0.2cm}Land-Share Agriculture & -0.

### most sensitivite parameter check

In [247]:
import copy

mse_param_result_set = []
modified_param_list = copy.deepcopy(param_list)
parameter_perturbation = 0.25
for idx, par in enumerate(param_list):
    value = par["values"][2] if isinstance(par["values"], list) else par["values"]

    if (
        "sigma_" in par["keys"]
        or "Lambda_" in par["keys"]
        or "V_" in par["keys"]
        or "tau_" in par["keys"]
    ):
        modified_param_list[idx]["values"] = value * (1 - parameter_perturbation)
        df_result = model.solve(modified_param_list)
        mse = (
            abs(df_baseline_model_result["mean value"] - df_result["mean value"])
        ).mean()
        wmse = abs(
            (df_result["mean value"] - df_baseline_model_result["mean value"])
            / df_baseline_model_result["mean value"]
        ).mean()
        mse_param_result_set.append(
            {"param": par["keys"], "mse": mse, "wmse": wmse, "end": "low"}
        )
        modified_param_list[idx]["values"] = value

        result_final = sum_pb_results(df_result)
        pb_unchanged = all((baseline_model_result_final > 0) == (result_final > 0))
        assert pb_unchanged

        assert all(
            [
                modified_param_list[i]["values"] == (p["values"])
                for i, p in enumerate(param_list)
                if isinstance(p, int)
            ]
        )
        assert all(
            [
                modified_param_list[i]["values"] == (p["values"][2])
                for i, p in enumerate(param_list)
                if isinstance(p, list)
            ]
        )

        modified_param_list[idx]["values"] = value * (1 + parameter_perturbation)
        df_result = model.solve(modified_param_list)
        mse = (
            abs(df_baseline_model_result["mean value"] - df_result["mean value"])
        ).mean()
        wmse = abs(
            (df_result["mean value"] - df_baseline_model_result["mean value"])
            / df_baseline_model_result["mean value"]
        ).mean()
        mse_param_result_set.append(
            {"param": par["keys"], "mse": mse, "wmse": wmse, "end": "high"}
        )
        modified_param_list[idx]["values"] = value

        result_final = sum_pb_results(df_result)
        pb_unchanged = all((baseline_model_result_final > 0) == (result_final > 0))
        assert pb_unchanged

        result_final = sum_pb_results(df_result)
        pb_change = all((baseline_model_result_final > 0) == (result_final > 0))
        assert all(
            [
                modified_param_list[i]["values"] == p["values"]
                for i, p in enumerate(param_list)
                if isinstance(p, int)
            ]
        )
        assert all(
            [
                modified_param_list[i]["values"] == (p["values"][2])
                for i, p in enumerate(param_list)
                if isinstance(p, list)
            ]
        )

        # print(par['keys'], par['values'])


mse_param_result_set.sort(key=lambda x: x["wmse"], reverse=True)
pd.DataFrame(mse_param_result_set)

,end,mse,param,wmse
0,low,0.0193,sigma_Eps,0.4505
1,high,0.0179,sigma_Eps,0.4355
2,low,0.0198,sigma_Y,0.2734
3,low,0.0007,sigma_nF,0.2547
4,high,0.0171,sigma_Y,0.2351
5,high,0.0005,sigma_nF,0.1906
6,low,0.0017,Lambda_M,0.1188
7,low,0.0047,sigma_nLA,0.1118
8,high,0.0045,sigma_nLA,0.1003
9,low,0.0141,Lambda_E,0.1001
